# Spotify RAG Application

Detta projekt bygger ett RAG-system (Retrieval Augmented Generation) med hjälp av Spotify-data.

## 1. Importerar bibliotek

In [23]:
import pandas as pd
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings 
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders.csv_loader import CSVLoader

## 2. Laddar API-nyckel


In [24]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

In [25]:
load_dotenv(dotenv_path='.env', override=True)

True

## 3. Läser in datasetet


In [26]:
csv_path = ("Data/cleaned_dataset.csv")

In [27]:
df = pd.read_csv("Data/cleaned_dataset.csv")

## 4. Undersöker datan


In [28]:
df.head()

,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## 5. Skapar embeddings-modell


In [29]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## 6. Skapar combined text


In [30]:
#Skapar text för embeddings 
df["text"] = (
    "Track name: " + df["track_name"].astype(str) +
    ". Artist: " + df["artists"].astype(str) +
    ". Album: " + df["album_name"].astype(str) +
    ". Genre: " + df["track_genre"].astype(str) +
    ". Popularity: " + df["popularity"].astype(str) +
    ". Danceability: " + df["danceability"].astype(str) +
    ". Energy: " + df["energy"].astype(str) +
    ". Tempo: " + df["tempo"].astype(str)
)

df["text"].head()

0    Track name: Comedy. Artist: Gen Hoshino. Album...
1    Track name: Ghost - Acoustic. Artist: Ben Wood...
2    Track name: To Begin Again. Artist: Ingrid Mic...
3    Track name: Can't Help Falling In Love. Artist...
4    Track name: Hold On. Artist: Chord Overstreet....
Name: text, dtype: str

## 7. Skapar dokument-objekt


In [31]:
# Skapar en lista av Document-objekt där varje objekt innehåller texten från "text"-kolumnen 
documents = []
for text in df["text"]:
    documents.append(Document(page_content=text))

len(documents)

113423

In [32]:
documents[0]

Document(metadata={}, page_content='Track name: Comedy. Artist: Gen Hoshino. Album: Comedy. Genre: acoustic. Popularity: 73. Danceability: 0.676. Energy: 0.461. Tempo: 87.917')

## 8. Begränsar datasetet


In [33]:
# Skapar en mindre lista av Document-objekt 
small_documents = documents[:30]
len(small_documents)

30

## 9. Skapar vektordatabasen


In [34]:
#skapar vecktordatabasen 
vectorstore = Chroma.from_documents(
    collection_name="Spotify",
    documents=small_documents,
    embedding=embeddings,
    persist_directory="chroma_spotify_db"
)

vectorstore.persist()

## 10. Skapar retriever


In [35]:
retriever = vectorstore.as_retriever()

## 11. Testar similarity search


In [36]:
results = retriever.invoke("What are some popular acoustic songs?")

results

[Document(metadata={}, page_content='Track name: Ghost - Acoustic. Artist: Ben Woodward. Album: Ghost (Acoustic). Genre: acoustic. Popularity: 55. Danceability: 0.42. Energy: 0.166. Tempo: 77.489'),
 Document(metadata={}, page_content='Track name: Ghost - Acoustic. Artist: Ben Woodward. Album: Ghost (Acoustic). Genre: acoustic. Popularity: 55. Danceability: 0.42. Energy: 0.166. Tempo: 77.489'),
 Document(metadata={}, page_content='Track name: Ghost - Acoustic. Artist: Ben Woodward. Album: Ghost (Acoustic). Genre: acoustic. Popularity: 55. Danceability: 0.42. Energy: 0.166. Tempo: 77.489'),
 Document(metadata={}, page_content='Track name: Ghost - Acoustic. Artist: Ben Woodward. Album: Ghost (Acoustic). Genre: acoustic. Popularity: 55. Danceability: 0.42. Energy: 0.166. Tempo: 77.489')]

## 12. Skapar prompt


In [37]:
#skapa en prompt 
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template(
    "You are a music expert. Based on the following retrieved information, answer the question: {context}\n\nQuestion: {question}"
)

In [38]:
question = "What are some popular acoustic songs?"

## 13. Laddar språkmodellen


In [44]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash-latest",
    temperature=0.7
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## 14. Genererar svar


In [ ]:
question = "What are some popular acoustic songs?"

docs = retriever.invoke(question)[:3]

context = "\n".join([doc.page_content for doc in docs])

final_prompt = prompt.format(
    context=context,
    question=question
)

response = llm.invoke(final_prompt)

print(response.content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-1.5-flash-latest' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash-latest is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}